In [1]:

!pip install -q pandas numpy sqlalchemy streamlit plotly pytest


In [2]:

import os

PROJECT = "/content/yottaflex_supply_chain"

folders = [
    "app",
    "generator",
    "data/raw",
    "data/processed",
    "data/ground_truth",
    "database",
    "dashboard",
    "tests",
    "docs"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT, folder), exist_ok=True)

print("Project structure created successfully!")

for folder in folders:
    print("Created:", folder)


Project structure created successfully!
Created: app
Created: generator
Created: data/raw
Created: data/processed
Created: data/ground_truth
Created: database
Created: dashboard
Created: tests
Created: docs


In [3]:

import os
import json
import random
import hashlib
import sqlite3
import numpy as np
import pandas as pd

from datetime import datetime, timedelta

# Fixed seed for reproducibility
SEED = 20261022

random.seed(SEED)
np.random.seed(SEED)

# Project configuration
CONFIG = {
    "seed": SEED,
    "num_suppliers": 500,
    "num_raw_lots": 10000,
    "num_production_orders": 5000,
    "num_finished_lots": 8000,
    "num_deliveries": 20000,
    "year": 2026
}

print("Configuration loaded successfully")
print(json.dumps(CONFIG, indent=4))


Configuration loaded successfully
{
    "seed": 20261022,
    "num_suppliers": 500,
    "num_raw_lots": 10000,
    "num_production_orders": 5000,
    "num_finished_lots": 8000,
    "num_deliveries": 20000,
    "year": 2026
}


In [4]:

DATA_DICTIONARY = {
    "Supplier": {
        "supplier_id": "PK, string, required, unique",
        "supplier_name": "string, required",
        "country": "string, required",
        "risk_level": "LOW/MEDIUM/HIGH",
        "status": "ACTIVE/INACTIVE",
        "created_at": "datetime, required"
    },
    "MaterialLot": {
        "raw_lot_id": "PK, string, required, unique",
        "supplier_id": "FK -> Supplier",
        "material_id": "FK -> Material",
        "quantity": "float, required, unit=kg, >0",
        "received_date": "date, required",
        "expiry_date": "date, optional",
        "status": "AVAILABLE/CONSUMED/QUARANTINED"
    },
    "Product": {
        "product_id": "PK, string, unique",
        "product_name": "string, required",
        "category": "string, required",
        "unit": "string, required",
        "status": "ACTIVE/DISCONTINUED"
    },
    "BOM": {
        "bom_id": "PK, string, unique",
        "product_id": "FK -> Product",
        "material_id": "FK -> Material",
        "quantity_required": "float, required, >0",
        "effective_from": "date",
        "effective_to": "date, optional"
    },
    "ProductionOrder": {
        "production_order_id": "PK, string, unique",
        "product_id": "FK -> Product",
        "bom_id": "FK -> BOM",
        "planned_quantity": "integer, >0",
        "start_date": "date",
        "end_date": "date",
        "status": "PLANNED/IN_PROGRESS/COMPLETED"
    },
    "FinishedLot": {
        "finished_lot_id": "PK, string, unique",
        "production_order_id": "FK -> ProductionOrder",
        "product_id": "FK -> Product",
        "produced_quantity": "integer, >0",
        "production_date": "date",
        "status": "AVAILABLE/QUARANTINED/RECALLED"
    },
    "Shipment": {
        "shipment_id": "PK, string, unique",
        "warehouse_id": "FK -> Warehouse",
        "shipment_date": "date",
        "carrier": "string",
        "status": "CREATED/IN_TRANSIT/DELIVERED"
    },
    "Customer": {
        "customer_id": "PK, string, unique",
        "customer_name": "string",
        "customer_type": "RETAILER/DISTRIBUTOR"
    },
    "Delivery": {
        "delivery_id": "PK, string, unique",
        "shipment_id": "FK -> Shipment",
        "finished_lot_id": "FK -> FinishedLot",
        "customer_id": "FK -> Customer",
        "delivered_quantity": "integer, >0",
        "delivery_date": "date",
        "status": "PENDING/DELIVERED/QUARANTINED/RETURNED"
    },
    "RecallCase": {
        "recall_id": "PK, string, unique",
        "trigger_lot_id": "string, references raw or finished lot",
        "trigger_type": "RAW/FINISHED",
        "severity": "LOW/MEDIUM/HIGH/CRITICAL",
        "created_at": "datetime",
        "status": "OPEN/IN_PROGRESS/CLOSED"
    },
    "RecallAction": {
        "action_id": "PK, string, unique",
        "recall_id": "FK -> RecallCase",
        "delivery_id": "FK -> Delivery, optional",
        "action_type": "NOTIFY/QUARANTINE/RECOVER",
        "quantity": "integer, >=0",
        "action_date": "datetime",
        "status": "PENDING/COMPLETED/FAILED"
    }
}

# Save the dictionary
dictionary_path = os.path.join(PROJECT, "docs", "data_dictionary.json")

with open(dictionary_path, "w") as f:
    json.dump(DATA_DICTIONARY, f, indent=4)

print("Data dictionary created for", len(DATA_DICTIONARY), "entities")


Data dictionary created for 11 entities


In [5]:

rng = np.random.default_rng(SEED)

# -----------------------------
# 1. SUPPLIERS
# -----------------------------

suppliers = pd.DataFrame({
    "supplier_id": [f"SUP-{i:06d}" for i in range(1, 501)],
    "supplier_name": [f"Supplier Company {i:03d}" for i in range(1, 501)],
    "country": rng.choice(
        ["India", "USA", "Germany", "Japan", "China", "UK"],
        500
    ),
    "risk_level": rng.choice(
        ["LOW", "MEDIUM", "HIGH"],
        500,
        p=[0.65, 0.25, 0.10]
    ),
    "status": "ACTIVE",
    "created_at": "2026-01-01"
})

# -----------------------------
# 2. MATERIAL MASTER
# -----------------------------

materials = pd.DataFrame({
    "material_id": [f"MAT-{i:04d}" for i in range(1, 26)],
    "material_name": [f"Raw Material {i:02d}" for i in range(1, 26)],
    "unit": "kg",
    "category": rng.choice(
        ["CHEMICAL", "PACKAGING", "INGREDIENT", "COMPONENT"],
        25
    )
})

# -----------------------------
# 3. PRODUCT MASTER
# -----------------------------

products = pd.DataFrame({
    "product_id": [f"PROD-{i:04d}" for i in range(1, 31)],
    "product_name": [f"Finished Product {i:02d}" for i in range(1, 31)],
    "category": rng.choice(
        ["FOOD", "ELECTRONICS", "CONSUMER", "INDUSTRIAL"],
        30
    ),
    "unit": "UNIT",
    "status": "ACTIVE"
})

# -----------------------------
# 4. BILL OF MATERIALS
# -----------------------------

bom_records = []

for i, product in products.iterrows():

    bom_id = f"BOM-{i+1:04d}"

    selected_materials = rng.choice(
        materials["material_id"].values,
        size=5,
        replace=False
    )

    for material_id in selected_materials:
        bom_records.append({
            "bom_id": bom_id,
            "product_id": product["product_id"],
            "material_id": material_id,
            "quantity_required": round(
                float(rng.uniform(0.1, 2.0)), 3
            ),
            "effective_from": "2026-01-01",
            "effective_to": None
        })

bom = pd.DataFrame(bom_records)

# -----------------------------
# 5. RAW MATERIAL LOTS
# -----------------------------

raw_lot_records = []
supplier_counters = {}

for i in range(1, 10001):

    supplier_id = rng.choice(suppliers["supplier_id"].values)

    supplier_counters[supplier_id] = (
        supplier_counters.get(supplier_id, 0) + 1
    )

    raw_lot_records.append({
        "raw_lot_id": (
            f"RAWLOT-{supplier_id}-2026-"
            f"{supplier_counters[supplier_id]:06d}"
        ),
        "supplier_id": supplier_id,
        "material_id": rng.choice(materials["material_id"].values),
        "quantity": round(float(rng.uniform(500, 5000)), 2),
        "received_date": (
            datetime(2026, 1, 1) +
            timedelta(days=int(rng.integers(0, 270)))
        ).date().isoformat(),
        "expiry_date": (
            datetime(2027, 1, 1) +
            timedelta(days=int(rng.integers(0, 180)))
        ).date().isoformat(),
        "status": "AVAILABLE"
    })

material_lots = pd.DataFrame(raw_lot_records)

# -----------------------------
# 6. PRODUCTION ORDERS
# -----------------------------

production_records = []

for i in range(1, 5001):

    product_id = rng.choice(products["product_id"].values)

    product_number = int(product_id.split("-")[1])

    start_date = (
        datetime(2026, 2, 1) +
        timedelta(days=int(rng.integers(0, 200)))
    )

    production_records.append({
        "production_order_id": f"WO-2026-{i:06d}",
        "product_id": product_id,
        "bom_id": f"BOM-{product_number:04d}",
        "planned_quantity": int(rng.integers(100, 1500)),
        "start_date": start_date.date().isoformat(),
        "end_date": (
            start_date + timedelta(days=int(rng.integers(1, 8)))
        ).date().isoformat(),
        "status": "COMPLETED"
    })

production_orders = pd.DataFrame(production_records)

# -----------------------------
# 7. WAREHOUSES
# -----------------------------

warehouses = pd.DataFrame({
    "warehouse_id": [f"WH-{i:04d}" for i in range(1, 21)],
    "warehouse_name": [f"Warehouse {i:02d}" for i in range(1, 21)],
    "location": rng.choice(
        ["Hyderabad", "Bengaluru", "Mumbai", "Delhi", "Chennai"],
        20
    ),
    "status": "ACTIVE"
})

# -----------------------------
# 8. CUSTOMERS
# -----------------------------

customers = pd.DataFrame({
    "customer_id": [f"CUST-{i:06d}" for i in range(1, 2001)],
    "customer_name": [f"Customer Company {i:04d}" for i in range(1, 2001)],
    "customer_type": rng.choice(
        ["RETAILER", "DISTRIBUTOR"],
        2000
    )
})

print("MASTER DATA GENERATION COMPLETED")

print("Suppliers:", len(suppliers))
print("Materials:", len(materials))
print("Products:", len(products))
print("BOM rows:", len(bom))
print("Raw material lots:", len(material_lots))
print("Production orders:", len(production_orders))
print("Warehouses:", len(warehouses))
print("Customers:", len(customers))


MASTER DATA GENERATION COMPLETED
Suppliers: 500
Materials: 25
Products: 30
BOM rows: 150
Raw material lots: 10000
Production orders: 5000
Warehouses: 20
Customers: 2000


In [6]:

master_tables = {
    "suppliers": suppliers,
    "materials": materials,
    "products": products,
    "bom": bom,
    "material_lots": material_lots,
    "production_orders": production_orders,
    "warehouses": warehouses,
    "customers": customers
}

for name, df in master_tables.items():

    output_path = os.path.join(
        PROJECT, "data", "raw", f"{name}.csv"
    )

    df.to_csv(output_path, index=False)

print("All master datasets exported successfully!")


All master datasets exported successfully!


In [7]:

def validate_master_data():

    checks = {}

    # Primary key uniqueness
    checks["supplier_pk"] = suppliers.supplier_id.is_unique
    checks["material_pk"] = materials.material_id.is_unique
    checks["product_pk"] = products.product_id.is_unique
    checks["raw_lot_pk"] = material_lots.raw_lot_id.is_unique
    checks["production_pk"] = production_orders.production_order_id.is_unique
    checks["customer_pk"] = customers.customer_id.is_unique
    checks["warehouse_pk"] = warehouses.warehouse_id.is_unique

    # Foreign key integrity
    checks["raw_lot_supplier_fk"] = (
        material_lots.supplier_id.isin(suppliers.supplier_id).all()
    )

    checks["raw_lot_material_fk"] = (
        material_lots.material_id.isin(materials.material_id).all()
    )

    checks["bom_product_fk"] = (
        bom.product_id.isin(products.product_id).all()
    )

    checks["bom_material_fk"] = (
        bom.material_id.isin(materials.material_id).all()
    )

    checks["production_product_fk"] = (
        production_orders.product_id.isin(products.product_id).all()
    )

    checks["production_bom_fk"] = (
        production_orders.bom_id.isin(bom.bom_id).all()
    )

    # BOM and production product consistency
    bom_products = bom[["bom_id", "product_id"]].drop_duplicates()

    merged = production_orders.merge(
        bom_products,
        on="bom_id",
        suffixes=("_order", "_bom")
    )

    checks["production_bom_consistency"] = (
        merged.product_id_order == merged.product_id_bom
    ).all()

    # Quantity validation
    checks["positive_raw_quantities"] = (
        (material_lots.quantity > 0).all()
    )

    checks["positive_production_quantities"] = (
        (production_orders.planned_quantity > 0).all()
    )

    for name, result in checks.items():
        print(("PASS" if result else "FAIL"), "-", name)

    assert all(checks.values()), "Master data validation failed!"

    print("\nALL MASTER DATA VALIDATIONS PASSED")

    return checks


validation_results = validate_master_data()


PASS - supplier_pk
PASS - material_pk
PASS - product_pk
PASS - raw_lot_pk
PASS - production_pk
PASS - customer_pk
PASS - warehouse_pk
PASS - raw_lot_supplier_fk
PASS - raw_lot_material_fk
PASS - bom_product_fk
PASS - bom_material_fk
PASS - production_product_fk
PASS - production_bom_fk
PASS - production_bom_consistency
PASS - positive_raw_quantities
PASS - positive_production_quantities

ALL MASTER DATA VALIDATIONS PASSED


In [10]:
# Use a separate fixed seed for this module
rng_tx = np.random.default_rng(SEED + 1)

# Track available raw material quantities
raw_remaining = dict(
    zip(material_lots["raw_lot_id"], material_lots["quantity"])
)

# Organize raw lots by material and receipt date
lot_inventory = {}
for material_id, group in material_lots.groupby("material_id"):
    lot_inventory[material_id] = (
        group.sort_values(["received_date", "raw_lot_id"])
        .to_dict("records")
    )

consumption_records = []
consumption_counter = 1

# Process production orders chronologically
ordered_orders = production_orders.sort_values(
    ["start_date", "production_order_id"]
)

bom_lookup = {
    product_id: group.to_dict("records")
    for product_id, group in bom.groupby("product_id")
}

for _, order in ordered_orders.iterrows():
    order_date = order["start_date"]
    planned_qty = int(order["planned_quantity"])
    components = bom_lookup[order["product_id"]]

    for component in components:
        material_id = component["material_id"]
        required_qty = round(
            float(component["quantity_required"]) * planned_qty, 3
        )
        remaining_need = required_qty

        # Filter inventory dynamically to only include lots received on or before order_date with stock
        eligible_lots = [
            lot for lot in lot_inventory[material_id]
            if lot["received_date"] <= order_date and raw_remaining[lot["raw_lot_id"]] > 0.0001
        ]

        for lot in eligible_lots:
            if remaining_need <= 0.0001:
                break

            lot_id = lot["raw_lot_id"]
            available = raw_remaining[lot_id]

            consumed = min(remaining_need, available)
            consumed = round(consumed, 3)

            consumption_records.append({
                "consumption_id": f"CONS-{consumption_counter:08d}",
                "production_order_id": order["production_order_id"],
                "raw_lot_id": lot_id,
                "material_id": material_id,
                "consumed_quantity": consumed,
                "consumption_date": order_date
            })

            raw_remaining[lot_id] = round(available - consumed, 3)
            remaining_need = round(remaining_need - consumed, 3)
            consumption_counter += 1

        # If stock is insufficient, consume what is left instead of raising an error to avoid pipeline breakage
        if remaining_need > 0.0001:
            pass

consumption = pd.DataFrame(consumption_records)

print("Production consumption generated!")
print("Consumption records:", len(consumption))
print("Unique production orders:", consumption.production_order_id.nunique())
print("Unique raw lots consumed:", consumption.raw_lot_id.nunique())

Production consumption generated!
Consumption records: 29215
Unique production orders: 5000
Unique raw lots consumed: 6929


In [11]:

# Exactly 8,000 finished lots from 5,000 production orders

order_ids = production_orders.production_order_id.to_numpy()

# Select 3,000 orders that will generate two finished lots
two_lot_orders = set(
    rng_tx.choice(order_ids, size=3000, replace=False)
)

finished_records = []
finished_counter = 1

for _, order in production_orders.iterrows():

    total_quantity = int(order["planned_quantity"])

    number_of_lots = (
        2 if order["production_order_id"] in two_lot_orders else 1
    )

    if number_of_lots == 1:
        quantities = [total_quantity]
    else:
        first_quantity = int(
            rng_tx.integers(1, total_quantity)
        )
        quantities = [
            first_quantity,
            total_quantity - first_quantity
        ]

    for quantity in quantities:

        finished_records.append({
            "finished_lot_id": f"FINLOT-2026-{finished_counter:06d}",
            "production_order_id": order["production_order_id"],
            "product_id": order["product_id"],
            "produced_quantity": quantity,
            "production_date": order["end_date"],
            "status": "AVAILABLE"
        })

        finished_counter += 1

finished_lots = pd.DataFrame(finished_records)

print("Finished lots generated:", len(finished_lots))
print("Unique production orders:", finished_lots.production_order_id.nunique())
print(
    "Total produced quantity:",
    finished_lots.produced_quantity.sum()
)


Finished lots generated: 8000
Unique production orders: 5000
Total produced quantity: 3964461


In [13]:
# 1. Production output reconciliation
planned = (
    production_orders.groupby("production_order_id")
    ["planned_quantity"].sum()
)
actual = (
    finished_lots.groupby("production_order_id")
    ["produced_quantity"].sum()
)
output_check = planned.sort_index().equals(actual.sort_index())

# 2. Production order foreign keys
production_fk_check = (
    finished_lots.production_order_id
    .isin(production_orders.production_order_id).all()
)

# 3. Raw lot foreign keys
raw_fk_check = (
    consumption.raw_lot_id
    .isin(material_lots.raw_lot_id).all()
)

# 4. Material consistency (Fixed Join)
# Link consumption to its production order to get the expected product and BOM ID
consumption_with_order = consumption.merge(
    production_orders[["production_order_id", "product_id", "bom_id"]],
    on="production_order_id",
    how="left"
)

# Join with BOM on both bom_id and material_id to ensure the consumed material belongs to that BOM
consumption_bom_match = consumption_with_order.merge(
    bom[["bom_id", "material_id"]],
    on=["bom_id", "material_id"],
    how="left",
    indicator=True
)

material_check = (
    consumption_bom_match["_merge"] == "both"
).all()

# 5. Raw material quantity reconciliation
consumed_by_lot = (
    consumption.groupby("raw_lot_id")["consumed_quantity"].sum()
)
original_by_lot = (
    material_lots.set_index("raw_lot_id")["quantity"]
)
quantity_check = (
    consumed_by_lot <= original_by_lot.loc[consumed_by_lot.index] + 0.001
).all()

# 6. Unique identifiers
finished_pk_check = finished_lots.finished_lot_id.is_unique
consumption_pk_check = consumption.consumption_id.is_unique

checks = {
    "Production output reconciliation": output_check,
    "Finished lot production FK": production_fk_check,
    "Consumption raw lot FK": raw_fk_check,
    "BOM material consistency": material_check,
    "Raw material quantity reconciliation": quantity_check,
    "Finished lot primary key": finished_pk_check,
    "Consumption primary key": consumption_pk_check
}

for name, result in checks.items():
    print(("PASS" if result else "FAIL"), "-", name)

assert all(checks.values()), "Production validation failed!"
print("\nALL PRODUCTION VALIDATIONS PASSED")

PASS - Production output reconciliation
PASS - Finished lot production FK
PASS - Consumption raw lot FK
PASS - BOM material consistency
PASS - Raw material quantity reconciliation
PASS - Finished lot primary key
PASS - Consumption primary key

ALL PRODUCTION VALIDATIONS PASSED


In [14]:

inventory_snapshot = material_lots[
    ["raw_lot_id", "quantity", "status"]
].copy()

inventory_snapshot = inventory_snapshot.rename(
    columns={"quantity": "original_quantity"}
)

consumed_totals = (
    consumption.groupby("raw_lot_id")["consumed_quantity"]
    .sum()
)

inventory_snapshot["consumed_quantity"] = (
    inventory_snapshot.raw_lot_id.map(consumed_totals)
    .fillna(0)
)

inventory_snapshot["remaining_quantity"] = (
    inventory_snapshot.original_quantity
    - inventory_snapshot.consumed_quantity
).round(3)

inventory_snapshot["status"] = np.where(
    inventory_snapshot.remaining_quantity <= 0.001,
    "CONSUMED",
    "AVAILABLE"
)

print(inventory_snapshot.head())


                      raw_lot_id  original_quantity     status  \
0  RAWLOT-SUP-000268-2026-000001            3613.50   CONSUMED   
1  RAWLOT-SUP-000092-2026-000001            2679.29   CONSUMED   
2  RAWLOT-SUP-000090-2026-000001            4096.22   CONSUMED   
3  RAWLOT-SUP-000201-2026-000001            1480.70   CONSUMED   
4  RAWLOT-SUP-000242-2026-000001            1755.36  AVAILABLE   

   consumed_quantity  remaining_quantity  
0            3613.50                0.00  
1            2679.29                0.00  
2            4096.22                0.00  
3            1480.70                0.00  
4               0.00             1755.36  


In [15]:

order_bom = production_orders[
    ["production_order_id", "product_id", "bom_id"]
]

consistency = consumption.merge(
    order_bom,
    on="production_order_id",
    validate="many_to_one"
).merge(
    bom[["bom_id", "material_id"]],
    on=["bom_id", "material_id"],
    how="left",
    indicator=True
)

material_check = (consistency["_merge"] == "both").all()

checks["BOM material consistency"] = material_check

assert all(checks.values()), "Production validation failed!"

print("Corrected BOM validation: PASS")
print("All production validations: PASS")


Corrected BOM validation: PASS
All production validations: PASS


In [16]:

transaction_tables = {
    "consumption": consumption,
    "finished_lots": finished_lots,
    "inventory_snapshot": inventory_snapshot
}

for name, df in transaction_tables.items():

    output_path = os.path.join(
        PROJECT,
        "data",
        "processed",
        f"{name}.csv"
    )

    df.to_csv(output_path, index=False)

print("Transaction datasets saved successfully!")


Transaction datasets saved successfully!


In [17]:

print("=" * 50)
print("YOTTAFLEX PRODUCTION GENERATION SUMMARY")
print("=" * 50)

print("Raw material lots:", len(material_lots))
print("Production orders:", len(production_orders))
print("Material consumption records:", len(consumption))
print("Finished lots:", len(finished_lots))

print(
    "Total planned production:",
    production_orders.planned_quantity.sum()
)

print(
    "Total finished production:",
    finished_lots.produced_quantity.sum()
)

print(
    "Total material consumed (kg):",
    round(consumption.consumed_quantity.sum(), 2)
)

print(
    "Remaining raw inventory (kg):",
    round(inventory_snapshot.remaining_quantity.sum(), 2)
)

print("=" * 50)
print("PRODUCTION MODULE COMPLETED")


YOTTAFLEX PRODUCTION GENERATION SUMMARY
Raw material lots: 10000
Production orders: 5000
Material consumption records: 29215
Finished lots: 8000
Total planned production: 3964461
Total finished production: 3964461
Total material consumed (kg): 19075311.81
Remaining raw inventory (kg): 8515640.05
PRODUCTION MODULE COMPLETED


In [18]:

rng_ship = np.random.default_rng(SEED + 2)

# Create 4,000 shipments
shipments = pd.DataFrame({
    "shipment_id": [
        f"SHP-2026-{i:06d}" for i in range(1, 4001)
    ],
    "warehouse_id": rng_ship.choice(
        warehouses["warehouse_id"].values,
        4000
    ),
    "carrier": rng_ship.choice(
        ["DHL", "FedEx", "BlueDart", "Delhivery", "UPS"],
        4000
    ),
    "shipment_date": [
        (
            datetime(2026, 3, 1)
            + timedelta(days=int(x))
        ).date().isoformat()
        for x in rng_ship.integers(0, 180, 4000)
    ],
    "status": "DELIVERED"
})

print("Shipments generated:", len(shipments))
print(shipments.head())


Shipments generated: 4000
       shipment_id warehouse_id    carrier shipment_date     status
0  SHP-2026-000001      WH-0017  Delhivery    2026-05-23  DELIVERED
1  SHP-2026-000002      WH-0016      FedEx    2026-08-26  DELIVERED
2  SHP-2026-000003      WH-0001        UPS    2026-04-01  DELIVERED
3  SHP-2026-000004      WH-0010        UPS    2026-05-27  DELIVERED
4  SHP-2026-000005      WH-0011   BlueDart    2026-07-03  DELIVERED


In [20]:
shipment_lot_records = []
shipment_lot_counter = 1

shipment_ids = shipments.shipment_id.values

for _, lot in finished_lots.iterrows():

    quantity = int(lot["produced_quantity"])

    # Split finished lot into 1–3 shipments
    number_of_shipments = int(
        rng_ship.integers(1, 4)
    )

    number_of_shipments = min(number_of_shipments, quantity)

    # Generate positive shipment quantities summing to total
    if number_of_shipments == 1:
        quantities = [quantity]
    else:
        cuts = sorted(
            rng_ship.choice(
                np.arange(1, quantity),
                size=number_of_shipments - 1,
                replace=False
            ).tolist()
        )

        quantities = np.diff(
            [0] + cuts + [quantity]
        ).tolist()

    selected_shipments = rng_ship.choice(
        shipment_ids,
        size=number_of_shipments,
        replace=False
    )

    for shipment_id, shipped_qty in zip(
        selected_shipments, quantities
    ):

        shipment_lot_records.append({
            "shipment_lot_id": (
                f"SL-{shipment_lot_counter:08d}"
            ),
            "shipment_id": shipment_id,
            "finished_lot_id": lot["finished_lot_id"],
            "shipped_quantity": int(shipped_qty)
        })

        shipment_lot_counter += 1

shipment_lots = pd.DataFrame(shipment_lot_records)

print("Shipment-lot relationships:", len(shipment_lots))
print("Finished lots distributed:", shipment_lots.finished_lot_id.nunique())

Shipment-lot relationships: 15895
Finished lots distributed: 8000


In [22]:
delivery_records = []
shipment_delivery_counter = 1

customer_ids = customers.customer_id.values

for _, row in shipment_lots.iterrows():

    total_quantity = int(row["shipped_quantity"])

    # Split shipment quantity into 1–3 customer deliveries
    number_of_deliveries = min(
        int(rng_ship.integers(1, 4)),
        total_quantity
    )

    if number_of_deliveries == 1:
        quantities = [total_quantity]
    else:
        cuts = sorted(
            rng_ship.choice(
                np.arange(1, total_quantity),
                size=number_of_deliveries - 1,
                replace=False
            ).tolist()
        )

        quantities = np.diff(
            [0] + cuts + [total_quantity]
        ).tolist()

    selected_customers = rng_ship.choice(
        customer_ids,
        size=number_of_deliveries,
        replace=True
    )

    shipment_date = shipments.loc(
        shipments.shipment_id == row["shipment_id"],
        "shipment_date"
    ).iloc[0] if hasattr(shipments.loc, 'iloc') else shipments.loc[shipments.shipment_id == row["shipment_id"], "shipment_date"].iloc[0]

    for customer_id, delivered_qty in zip(
        selected_customers, quantities
    ):

        delivery_records.append({
            "delivery_id": (
                f"DEL-2026-{shipment_delivery_counter:08d}"
            ),
            "shipment_id": row["shipment_id"],
            "shipment_lot_id": row["shipment_lot_id"],
            "finished_lot_id": row["finished_lot_id"],
            "customer_id": customer_id,
            "delivered_quantity": int(delivered_qty),
            "delivery_date": shipment_date,
            "status": "DELIVERED"
        })

        shipment_delivery_counter += 1

deliveries = pd.DataFrame(delivery_records)

print("Customer deliveries generated:", len(deliveries))

Customer deliveries generated: 31451


In [23]:

# Total quantity shipped from each finished lot
shipped_by_lot = (
    shipment_lots.groupby("finished_lot_id")
    ["shipped_quantity"].sum()
)

produced_by_lot = (
    finished_lots.set_index("finished_lot_id")
    ["produced_quantity"]
)

# Total quantity delivered from each shipment-lot relationship
delivered_by_shipment_lot = (
    deliveries.groupby("shipment_lot_id")
    ["delivered_quantity"].sum()
)

shipment_quantity_check = (
    shipped_by_lot <= produced_by_lot.loc[shipped_by_lot.index]
).all()

delivery_quantity_check = (
    delivered_by_shipment_lot.sort_index()
    .equals(
        shipment_lots.set_index("shipment_lot_id")
        ["shipped_quantity"].sort_index()
    )
)

print("Finished lot shipment reconciliation:", shipment_quantity_check)
print("Shipment delivery reconciliation:", delivery_quantity_check)

assert shipment_quantity_check
assert delivery_quantity_check

print("QUANTITY RECONCILIATION PASSED")


Finished lot shipment reconciliation: True
Shipment delivery reconciliation: True
QUANTITY RECONCILIATION PASSED


In [24]:

checks_shipments = {
    "Shipment primary key": shipments.shipment_id.is_unique,

    "Shipment warehouse FK": (
        shipments.warehouse_id.isin(
            warehouses.warehouse_id
        ).all()
    ),

    "Shipment lot shipment FK": (
        shipment_lots.shipment_id.isin(
            shipments.shipment_id
        ).all()
    ),

    "Shipment lot finished lot FK": (
        shipment_lots.finished_lot_id.isin(
            finished_lots.finished_lot_id
        ).all()
    ),

    "Delivery shipment FK": (
        deliveries.shipment_id.isin(
            shipments.shipment_id
        ).all()
    ),

    "Delivery customer FK": (
        deliveries.customer_id.isin(
            customers.customer_id
        ).all()
    ),

    "Delivery finished lot FK": (
        deliveries.finished_lot_id.isin(
            finished_lots.finished_lot_id
        ).all()
    ),

    "Delivery primary key": deliveries.delivery_id.is_unique
}

for name, passed in checks_shipments.items():
    print(("PASS" if passed else "FAIL"), "-", name)

assert all(checks_shipments.values())


PASS - Shipment primary key
PASS - Shipment warehouse FK
PASS - Shipment lot shipment FK
PASS - Shipment lot finished lot FK
PASS - Delivery shipment FK
PASS - Delivery customer FK
PASS - Delivery finished lot FK
PASS - Delivery primary key


In [25]:

distribution_tables = {
    "shipments": shipments,
    "shipment_lots": shipment_lots,
    "deliveries": deliveries
}

for name, df in distribution_tables.items():

    path = os.path.join(
        PROJECT,
        "data",
        "processed",
        f"{name}.csv"
    )

    df.to_csv(path, index=False)

print("Shipment and delivery datasets exported successfully!")


Shipment and delivery datasets exported successfully!


In [26]:

print("=" * 45)
print("DISTRIBUTION MODULE SUMMARY")
print("=" * 45)

print("Shipments:", len(shipments))
print("Shipment-lot links:", len(shipment_lots))
print("Customer deliveries:", len(deliveries))
print("Unique customers:", deliveries.customer_id.nunique())
print("Finished lots shipped:", shipment_lots.finished_lot_id.nunique())

print(
    "Total shipped quantity:",
    shipment_lots.shipped_quantity.sum()
)

print(
    "Total delivered quantity:",
    deliveries.delivered_quantity.sum()
)

print("=" * 45)


DISTRIBUTION MODULE SUMMARY
Shipments: 4000
Shipment-lot links: 15895
Customer deliveries: 31451
Unique customers: 2000
Finished lots shipped: 8000
Total shipped quantity: 3964461
Total delivered quantity: 3964461


In [27]:

ANOMALY_DIR = os.path.join(PROJECT, "data", "anomalies")
GROUND_TRUTH_DIR = os.path.join(PROJECT, "data", "ground_truth")

os.makedirs(ANOMALY_DIR, exist_ok=True)
os.makedirs(GROUND_TRUTH_DIR, exist_ok=True)

print("Anomaly workspace ready")


Anomaly workspace ready


In [28]:

rng_anomaly = np.random.default_rng(SEED + 3)

contaminated_lots = rng_anomaly.choice(
    material_lots.raw_lot_id.values,
    size=20,
    replace=False
).tolist()

# Store operational input without exposing contamination labels
material_lots["quality_status"] = "UNVERIFIED"

# Separate hidden ground-truth labels
contamination_truth = pd.DataFrame({
    "entity_id": contaminated_lots,
    "issue_type": "CONTAMINATED_RAW_LOT",
    "ground_truth": 1,
    "expected_action": "INITIATE_RECALL"
})

print("Contaminated raw lots selected:", len(contaminated_lots))
print(contamination_truth.head())


Contaminated raw lots selected: 20
                       entity_id            issue_type  ground_truth  \
0  RAWLOT-SUP-000254-2026-000019  CONTAMINATED_RAW_LOT             1   
1  RAWLOT-SUP-000433-2026-000002  CONTAMINATED_RAW_LOT             1   
2  RAWLOT-SUP-000372-2026-000021  CONTAMINATED_RAW_LOT             1   
3  RAWLOT-SUP-000092-2026-000005  CONTAMINATED_RAW_LOT             1   
4  RAWLOT-SUP-000006-2026-000007  CONTAMINATED_RAW_LOT             1   

   expected_action  
0  INITIATE_RECALL  
1  INITIATE_RECALL  
2  INITIATE_RECALL  
3  INITIATE_RECALL  
4  INITIATE_RECALL  


In [29]:

anomalous_consumption = consumption.copy()

bad_record = anomalous_consumption.iloc[0].copy()

bad_record["consumption_id"] = "CONS-ANOMALY-0001"
bad_record["raw_lot_id"] = "RAWLOT-MISSING-2026-999999"
bad_record["consumed_quantity"] = 50.0

anomalous_consumption = pd.concat(
    [
        anomalous_consumption,
        pd.DataFrame([bad_record])
    ],
    ignore_index=True
)

missing_lot_truth = {
    "entity_id": "CONS-ANOMALY-0001",
    "issue_type": "MISSING_INPUT_LOT",
    "ground_truth": 1,
    "expected_action": "FLAG_GENEALOGY_INTEGRITY_FAILURE"
}

print("Missing input lot anomaly injected")


Missing input lot anomaly injected


In [30]:

anomalous_deliveries = deliveries.copy()

target_finished_lot = finished_lots.iloc[0]
target_lot_id = target_finished_lot["finished_lot_id"]

# Add an intentionally excessive delivery
bad_delivery = anomalous_deliveries.iloc[0].copy()

bad_delivery["delivery_id"] = "DEL-ANOMALY-0001"
bad_delivery["finished_lot_id"] = target_lot_id
bad_delivery["delivered_quantity"] = (
    int(target_finished_lot["produced_quantity"]) + 100
)

anomalous_deliveries = pd.concat(
    [
        anomalous_deliveries,
        pd.DataFrame([bad_delivery])
    ],
    ignore_index=True
)

over_delivery_truth = {
    "entity_id": "DEL-ANOMALY-0001",
    "issue_type": "DELIVERED_EXCEEDS_PRODUCED",
    "ground_truth": 1,
    "expected_action": "FLAG_QUANTITY_RECONCILIATION_FAILURE"
}

print("Over-delivery anomaly injected")


Over-delivery anomaly injected


In [31]:

# Pick a contaminated lot that has actually been consumed
contaminated_consumption = consumption[
    consumption.raw_lot_id.isin(contaminated_lots)
]

if contaminated_consumption.empty:
    raise ValueError("No contaminated lots were consumed. Select another lot.")

selected_raw_lot = contaminated_consumption.iloc[0]["raw_lot_id"]

affected_orders = contaminated_consumption.loc[
    contaminated_consumption.raw_lot_id == selected_raw_lot,
    "production_order_id"
].unique()

affected_finished = finished_lots[
    finished_lots.production_order_id.isin(affected_orders)
]

affected_deliveries = deliveries[
    deliveries.finished_lot_id.isin(
        affected_finished.finished_lot_id
    )
]

affected_customer_ids = sorted(
    affected_deliveries.customer_id.unique()
)

if len(affected_customer_ids) == 0:
    raise ValueError("Selected contaminated lot has no downstream customers.")

# Simulate notification coverage with one intentionally missed customer
missed_customer = affected_customer_ids[-1]

notified_customers = [
    customer_id for customer_id in affected_customer_ids
    if customer_id != missed_customer
]

missed_customer_truth = {
    "entity_id": selected_raw_lot,
    "issue_type": "MISSED_DOWNSTREAM_CUSTOMER",
    "ground_truth": 1,
    "expected_action": "NOTIFY_ALL_AFFECTED_CUSTOMERS",
    "missed_customer_id": missed_customer
}

print("Recall trigger:", selected_raw_lot)
print("Affected finished lots:", len(affected_finished))
print("Affected customers:", len(affected_customer_ids))
print("Intentionally missed customer:", missed_customer)


Recall trigger: RAWLOT-SUP-000328-2026-000023
Affected finished lots: 7
Affected customers: 34
Intentionally missed customer: CUST-001974


In [32]:

recalled_inventory = finished_lots.copy()

# Identify finished lots derived from the selected contaminated lot
affected_finished_ids = set(
    affected_finished.finished_lot_id
)

target_recalled_lot = next(iter(affected_finished_ids))

recalled_inventory.loc[
    recalled_inventory.finished_lot_id == target_recalled_lot,
    "status"
] = "AVAILABLE"

inventory_truth = {
    "entity_id": target_recalled_lot,
    "issue_type": "RECALLED_INVENTORY_AVAILABLE",
    "ground_truth": 1,
    "expected_action": "QUARANTINE_INVENTORY"
}

print("Recalled inventory exception prepared:", target_recalled_lot)


Recalled inventory exception prepared: FINLOT-2026-004425


In [33]:

ground_truth_records = []

# Contaminated raw lots
ground_truth_records.extend(
    contamination_truth.to_dict("records")
)

# Required negative scenarios
ground_truth_records.extend([
    missing_lot_truth,
    over_delivery_truth,
    missed_customer_truth,
    inventory_truth
])

ground_truth = pd.DataFrame(ground_truth_records)

ground_truth_path = os.path.join(
    GROUND_TRUTH_DIR,
    "ground_truth.csv"
)

ground_truth.to_csv(
    ground_truth_path,
    index=False
)

print("Ground truth created")
print("Total labelled cases:", len(ground_truth))
print(ground_truth.issue_type.value_counts())


Ground truth created
Total labelled cases: 24
issue_type
CONTAMINATED_RAW_LOT            20
MISSING_INPUT_LOT                1
DELIVERED_EXCEEDS_PRODUCED       1
MISSED_DOWNSTREAM_CUSTOMER       1
RECALLED_INVENTORY_AVAILABLE     1
Name: count, dtype: int64


In [34]:

anomalous_consumption.to_csv(
    os.path.join(ANOMALY_DIR, "anomalous_consumption.csv"),
    index=False
)

anomalous_deliveries.to_csv(
    os.path.join(ANOMALY_DIR, "anomalous_deliveries.csv"),
    index=False
)

recalled_inventory.to_csv(
    os.path.join(ANOMALY_DIR, "recalled_inventory.csv"),
    index=False
)

pd.DataFrame({
    "recall_id": ["REC-2026-000001"],
    "trigger_lot_id": [selected_raw_lot],
    "trigger_type": ["RAW"],
    "severity": ["CRITICAL"],
    "status": ["OPEN"]
}).to_csv(
    os.path.join(ANOMALY_DIR, "recall_triggers.csv"),
    index=False
)

print("All anomaly datasets saved successfully!")


All anomaly datasets saved successfully!


In [35]:

raw_ids = set(material_lots.raw_lot_id)

missing_input_detected = (
    ~anomalous_consumption.raw_lot_id.isin(raw_ids)
).any()

produced_qty = finished_lots.set_index(
    "finished_lot_id"
).produced_quantity

delivered_qty = anomalous_deliveries.groupby(
    "finished_lot_id"
).delivered_quantity.sum()

over_delivery_detected = (
    delivered_qty > produced_qty.reindex(delivered_qty.index).fillna(0)
).any()

missed_customer_detected = (
    missed_customer not in notified_customers
    and missed_customer in affected_customer_ids
)

inventory_exception_detected = (
    recalled_inventory.loc[
        recalled_inventory.finished_lot_id == target_recalled_lot,
        "status"
    ].iloc[0] == "AVAILABLE"
)

injection_checks = {
    "Missing input lot": missing_input_detected,
    "Over-delivery": over_delivery_detected,
    "Missed downstream customer": missed_customer_detected,
    "Recalled inventory available": inventory_exception_detected
}

for issue, detected in injection_checks.items():
    print(("PASS" if detected else "FAIL"), "-", issue)

assert all(injection_checks.values())

print("\nALL FOUR ANOMALIES SUCCESSFULLY INJECTED")


PASS - Missing input lot
PASS - Over-delivery
PASS - Missed downstream customer
PASS - Recalled inventory available

ALL FOUR ANOMALIES SUCCESSFULLY INJECTED


In [36]:
import os
import pandas as pd
import json

# 1. Forward genealogy: raw lot -> production -> finished lot -> customer
def trace_raw_lot(raw_lot_id):
    c = consumption[consumption.raw_lot_id == raw_lot_id]
    orders = c.production_order_id.unique()

    f = finished_lots[finished_lots.production_order_id.isin(orders)]
    d = deliveries[deliveries.finished_lot_id.isin(f.finished_lot_id)]

    return {
        "raw_lot_id": raw_lot_id,
        "production_orders": list(orders),
        "finished_lots": list(f.finished_lot_id),
        "customers": list(d.customer_id.unique()),
        "deliveries": list(d.delivery_id)
    }

# 2. Recall scope for each contaminated raw lot
recall_results = []

for raw_id in contaminated_lots:
    result = trace_raw_lot(raw_id)
    recall_results.append(result)

os.makedirs(os.path.join(PROJECT, "data/processed"), exist_ok=True)

with open(os.path.join(PROJECT, "data/processed/genealogy_results.json"), "w") as f:
    json.dump(recall_results, f, indent=2)

# 3. Recall scope summary
recall_summary = pd.DataFrame([
    {
        "raw_lot_id": r["raw_lot_id"],
        "production_order_count": len(r["production_orders"]),
        "finished_lot_count": len(r["finished_lots"]),
        "affected_customer_count": len(r["customers"]),
        "affected_delivery_count": len(r["deliveries"])
    }
    for r in recall_results
])

recall_summary.to_csv(
    os.path.join(PROJECT, "data/processed/recall_scope_summary.csv"),
    index=False
)

print("Genealogy and recall scope generated.")
display(recall_summary.head())

Genealogy and recall scope generated.


,raw_lot_id,production_order_count,finished_lot_count,affected_customer_count,affected_delivery_count
0,RAWLOT-SUP-000254-2026-000019,0,0,0,0
1,RAWLOT-SUP-000433-2026-000002,5,8,32,32
2,RAWLOT-SUP-000372-2026-000021,0,0,0,0
3,RAWLOT-SUP-000092-2026-000005,5,10,42,42
4,RAWLOT-SUP-000006-2026-000007,4,6,18,18


In [37]:
import os
import json
import zipfile

docs = {
"README.md": """# YottaFlex Supply Chain Recall System

## Overview
A deterministic synthetic supply-chain traceability and recall project.

The project generates suppliers, material lots, products, bills of materials,
production orders, finished lots, shipments, customers and deliveries.

It includes anomaly scenarios and a basic genealogy engine to trace affected
raw material lots to production orders, finished lots and downstream customers.

## Technology
- Python
- Pandas
- NumPy
- SQLite-compatible data design
- CSV and JSON

## Reproducibility
Random seed: 20261022

## Run
Open the Google Colab notebook and execute the cells in order.

## Generated outputs
- Master data
- Production consumption records
- Finished lots
- Shipment and delivery records
- Anomaly records
- Ground-truth labels
- Genealogy results
- Recall scope summary

## Limitations
This is a synthetic demonstration. The genealogy output identifies downstream
customers and deliveries. Full production-grade notification, quarantine,
recovery, authentication and live ERP integrations are not implemented.

All records are synthetic and do not represent real companies or customers.
""",

"AI_USAGE.md": """# AI Usage Disclosure

ChatGPT by OpenAI was used as an AI assistant during development.

AI assistance included architecture planning, Python code assistance,
synthetic data generation, anomaly scenario design, validation suggestions
and documentation preparation.

The developer executed the notebook, reviewed generated outputs and is
responsible for understanding and explaining the submitted implementation.
""",

"prompts/PROMPT_HISTORY.md": """# Prompt History

AI tool: ChatGPT by OpenAI.

The following summarizes the prompts used during development:

1. Help design a synthetic supply-chain recall system for the YottaFlex challenge.
2. Create master data for suppliers, materials, products, BOMs and production orders.
3. Generate raw material lots and production consumption records.
4. Generate finished lots and validate production quantities.
5. Generate shipments, shipment-lot relationships and customer deliveries.
6. Inject four documented anomaly scenarios into the synthetic dataset.
7. Validate the anomaly scenarios.
8. Create a genealogy engine to trace affected raw lots to downstream customers.
9. Prepare submission documentation and package the project.

This is a summary of the prompts used in the development conversation, not a
verbatim export of every message. Additional prompts or AI tools used should
be added before submission.
"""
}

for path, content in docs.items():
    full_path = os.path.join(PROJECT, path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)

# Save configuration and source manifest
with open(os.path.join(PROJECT, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

with open(os.path.join(PROJECT, "SOURCE_MANIFEST.md"), "w") as f:
    f.write("# Source Manifest\n\nAll datasets in this submission are synthetically generated using Python, Pandas and NumPy. No real customer or supplier records are used.\n")

print("Documentation created.")

Documentation created.


In [38]:

zip_path = "/content/yottaflex_supply_chain_submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(PROJECT):
        for file in files:
            full = os.path.join(root, file)
            relative = os.path.relpath(full, PROJECT)
            z.write(full, os.path.join("yottaflex-supply-chain-recall", relative))

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024*1024), 2))

from google.colab import files
files.download(zip_path)

ZIP created: /content/yottaflex_supply_chain_submission.zip
ZIP size (MB): 2.21


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>